# Chess-Coach Agent - Gemma 4 E4B QLoRA on Kaggle (1 or 2x T4)

LoRA adapter on the v1.2 agentic-harness corpus (skills + tools + the `python` verify
tool; fast/think/auto). Download the adapter -> serve as GGUF on the RTX 4060.

## GPU options
E4B 4-bit fits ONE 16GB T4 at seq 1664 (gradient checkpointing makes it fit; the earlier
OOM saga was a self-inflicted bug). Two ways to use the hardware:
- **DDP=True (2x T4, ~2x faster):** `accelerate launch` runs ONE full E4B replica per GPU,
  each trains a disjoint data shard, grads all-reduced each step. NOT model sharding.
- **DDP=False (1x T4):** plain single-GPU; uses one of the two cards.

## Setup
1. Settings -> Accelerator -> GPU T4 x2 (needed for DDP). 2. Internet On.
3. Add-ons -> Secrets -> HF_TOKEN (WRITE scope — needed to push checkpoints).
4. Set `CKPT_REPO` in Cell 1 to a private HF repo id (e.g. `you/gemma4-chesscoach-ckpt`).
5. Accept the Gemma 4 license once. Run top->bottom. After Cell 4 (deps) restart if
   prompted. Cell 6.5 (fit) + 6.55 (DDP probe) are the go/no-go before the long run.

## Multi-session resume = AUTOMATIC (12h ceiling / multiple accounts)
A full epoch is ~30h > Kaggle's 12h cap, so it spans ~3 sessions. With `CKPT_REPO` set,
the trainer mirrors checkpoint/ + best/ to the Hub every SAVE_EVERY steps, and **Cell 6.6
auto-pulls the latest checkpoint each session and continues** — no RESUME flag, no
download, no Kaggle Dataset. Just re-run the notebook top-to-bottom every session until
`upd N == MAX_STEPS`. (The Cell-1 RESUME/RESUME_DATASET vars are only a legacy fallback
for when CKPT_REPO is empty.) A startup self-test (Cell 7 logs `[hub] self-test OK`)
verifies the mirror works in the first minute, before any steps are invested.

## OOM ladder (never cut seq - it's the data floor)
RANK 16->8 -> keep TARGETS attn-only -> DDP=False (1 GPU). seq stays 1664.


In [ ]:
# Cell 1 — config (E4B QLoRA on Kaggle: 1×T4 normal, or 2×T4 for faster DDP)
import os

MODEL  = "gemma4_e4b"   # E4B FITS one T4 @ seq 1664 once gradient checkpointing works
ENGINE = "unsloth"   # Gemma-4-native; DDP runs one replica per GPU (set DDP=True below).

HF_REPO = {
    ("gemma4_e4b", "unsloth"): "unsloth/gemma-4-E4B-it",
    ("gemma4_e2b", "unsloth"): "unsloth/gemma-4-E2B-it",
    ("gemma4_e4b", "cuda"):    "google/gemma-4-E4B-it",
    ("gemma4_e2b", "cuda"):    "google/gemma-4-E2B-it",
}[(MODEL, ENGINE)]
NO_4BIT = False  # E4B MUST be 4-bit

CHESS_REPO_URL = "https://github.com/RyanDev1st/llm-and-engine.git"
CHESS_BRANCH = "feat/chess-coach-sft"
WORKDIR = "/kaggle/working/llm-and-engine"
OUTPUT = "gemma4_chess_e4b_kaggle"
DATA_STEM = "v1_2"   # FULL 73k: max routing coverage (V1_O 25%). lean=v1_2_lean (53.8k).

# SEQ = data floor (corpus max 1642, median 1305). Below 1664 truncates finals. Fixed.
MAX_SEQ = 1664   # E4B 4-bit @1664 fits ONE T4 at ~10GB (user-tested) once gradient
                 # checkpointing fires. DDP / 2-GPU is only for ~2x speed, never required.
TARGETS = "attn-only"    # OOM ladder: attn-only + rank 16 -> rank 8 -> never cut seq.
RANK = 16
GRAD_ACCUM = 16
# DDP: True -> launch 2 processes (one full E4B replica per T4, disjoint data
# shards, grads all-reduced) for ~2x throughput. Needs Accelerator GPU T4 x2.
# False -> single-GPU (uses 1 of the 2 T4s). Both work; DDP just trains faster.
DDP = True

# --- FULL EPOCH = ~30h on 2xT4 (2283 updates x grad_accum 16). Kaggle caps a session
# at 12h, so 1 epoch spans ~3 sessions. SET CKPT_REPO below so a 12h SIGKILL can't lose
# progress (the in-process crash-save can't catch a SIGKILL; a timed-out commit's local
# Output is not guaranteed). best/ persists -> you can stop anytime and serve best. ---
MAX_STEPS = 2283   # ~1 full epoch under DDP-2 (use 4566 if DDP=False).
EVAL_EVERY = 200         # eval ~4min each; every 200 updates keeps best/ fresh, low overhead
MAX_VAL = 128
SAVE_EVERY = 50          # checkpoint adapter+optimizer every 50 steps (crash/timeout-safe + resume)

# --- MULTI-SESSION RESUME = AUTOMATIC. With CKPT_REPO set, Cell 6.6 auto-pulls the latest
# checkpoint from the Hub each session and continues — NO RESUME flag, NO download, NO
# Kaggle Dataset. Just re-run the notebook top-to-bottom every session. (The two vars below
# are only the LEGACY Dataset fallback used when CKPT_REPO is empty; Cell 6.6 sets RESUME.)
RESUME = False           # auto-managed by Cell 6.6; leave False
RESUME_DATASET = "/kaggle/input/REPLACE-with-your-checkpoint-dataset"  # (legacy) only if CKPT_REPO==""

# OFF-KERNEL CHECKPOINT (required for the ~30h/3-session epoch): push every checkpoint+best
# to a PRIVATE HF Hub repo so a 12h SIGKILL can't lose progress AND resume is automatic
# across sessions/accounts. Fill with your HF repo id; leave "" to disable (local only).
CKPT_REPO = ""   # e.g. "RyanDev1st/gemma4-chesscoach-ckpt"
if CKPT_REPO:
    os.environ["CHESS_CKPT_REPO"] = CKPT_REPO   # picked up by train_unsloth _hub_push
    print("off-kernel checkpoint repo:", CKPT_REPO)

print("config:", MODEL, "engine=", ENGINE, "base=", HF_REPO, "seq=", MAX_SEQ,
      "rank=", RANK, "targets=", TARGETS, "steps=", MAX_STEPS, "ckpt_repo=", CKPT_REPO or "(local only)")


In [ ]:
# Cell 2 — GPU preflight (E4B fits ONE T4 @1664 ~10GB; T4×2 only buys faster DDP)
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                      "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU — Settings → Accelerator → GPU T4 (×2 for DDP)"
n = torch.cuda.device_count()
print("cuda", torch.version.cuda, "| device", torch.cuda.get_device_name(0), "| count", n)
if n < 2:
    print("\nℹ️  Only 1 GPU — fine, this is a normal run. E4B@1664 fits ONE T4 at ~10GB\n"
          "    (tested). Set DDP=False in Cell 1. Use T4×2 only for the ~2x DDP speedup.")


In [ ]:
# Cell 3 — clone repo (skip LFS weights; we only need code + data)
import os, subprocess

def run(cmd, **kw):
    print(">", " ".join(cmd)); return subprocess.run(cmd, check=True, text=True, **kw)

env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
if not os.path.exists(WORKDIR):
    run(["git", "clone", "--depth", "1", "--branch", CHESS_BRANCH, CHESS_REPO_URL, WORKDIR], env=env)
else:
    run(["git", "-C", WORKDIR, "pull", "--ff-only"], env=env)
run(["git", "-C", WORKDIR, "log", "-1", "--oneline"])
os.environ["PYTHONPATH"] = f"{WORKDIR}/src/llm"
print("PYTHONPATH=", os.environ["PYTHONPATH"])


In [ ]:
# Cell 4 — deps. Let Unsloth resolve its own current stack (do NOT pin an old
# transformers — Gemma 4 needs a recent one).
import subprocess, sys
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)

if ENGINE == "unsloth":
    pip("--upgrade", "unsloth", "unsloth_zoo", "bitsandbytes")
    pip("python-chess")
    print("unsloth installed. If Kaggle shows a RESTART banner, restart, then run Cells 5+. "
          "If Gemma 4 fails to load with a model-type error, run: pip install -U transformers")
else:
    pip("-U", "transformers", "accelerate", "peft", "trl",
        "bitsandbytes", "datasets", "sentencepiece", "protobuf", "python-chess")
    import transformers, peft, bitsandbytes
    print("transformers", transformers.__version__, "| peft", peft.__version__,
          "| bnb", bitsandbytes.__version__)


In [ ]:
# Cell 5 — HF login + download base model (token from Kaggle Secrets)
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, snapshot_download

login(UserSecretsClient().get_secret("HF_TOKEN"))
dest = f"{WORKDIR}/src/llm/models/{MODEL}"
snapshot_download(repo_id=HF_REPO, local_dir=dest,
                  allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt", "tokenizer*"])
print("base model at", dest)
print(sorted(os.listdir(dest)))


In [ ]:
# Cell 6 - data check (must be the REGENERATED grounded corpus; stored gzipped)
import os, gzip
for name in (f"{DATA_STEM}_train.jsonl", f"{DATA_STEM}_val.jsonl"):
    base = f"{WORKDIR}/data/sft/{name}"
    path = base if os.path.exists(base) else base + ".gz"
    if not os.path.exists(path):
        n = 0
    elif path.endswith(".gz"):
        with gzip.open(path, "rt", encoding="utf-8") as fh:
            n = sum(1 for _ in fh)
    else:
        n = sum(1 for _ in open(path, encoding="utf-8"))
    print(name, "rows=", n, "(", os.path.basename(path), ")")
    assert n > 0, f"missing/empty {path} - commit the regenerated corpus to the branch first"


In [ ]:
# Cell 6.5 - FIT TEST (go/no-go): real Unsloth path, 3 steps, 64 rows, BOTH GPUs.
# True memory profile (the old single-GPU HF probe was heavier+misleading - ignore it).
# Survives 3 steps -> run Cell 7. OOM -> climb the Cell-1 ladder (RANK 16->8). Never cut seq.
import subprocess, sys, os
cmd = [sys.executable, "-m", "llm_training.run_train",
       "--model", MODEL, "--engine", ENGINE, "--max-seq", str(MAX_SEQ),
       "--rank", str(RANK), "--targets", TARGETS, "--grad-accum", "1",
       "--max-steps", "3", "--max-examples", "64", "--max-val", "0", "--data-stem", DATA_STEM, "--output", "fit_test"]
print(">", " ".join(cmd))
r = subprocess.run(cmd, cwd=WORKDIR, capture_output=True, text=True,
    env={**os.environ, "PYTHONPATH": f"{WORKDIR}/src/llm",
         "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
         "TORCHDYNAMO_DISABLE": "1", "UNSLOTH_COMPILE_DISABLE": "1"})
print("RC", r.returncode)
print("STDOUT tail:")
print(r.stdout[-2000:])
if r.returncode != 0:
    print("STDERR tail:")
    print(r.stderr[-4000:])
    raise SystemExit("fit test FAILED - read STDERR. If CUDA OOM: RANK 16->8, re-run. "
                     "Confirm Cell 2 shows count=2 (balanced needs 2 GPUs).")
print("FITS on the 2xT4 sharded path -> run Cell 7. Note the s/step printed above.")


In [ ]:
# Cell 6.55 - DDP fit + THROUGHPUT A/B (real Unsloth path). Replaces the old plain-HF
# probe (that one OOM'd misleadingly). Runs the REAL trainer at 2 GPUs vs 1 GPU, 8 steady
# steps each (grad-accum 1), parses s/step, reports the effective speedup. Skip if DDP=False.
import subprocess, os, re, statistics
def _run(nproc):
    args=['--model',MODEL,'--engine','unsloth','--max-seq',str(MAX_SEQ),'--rank',str(RANK),
          '--targets',TARGETS,'--grad-accum','1','--max-steps','8','--max-examples','64',
          '--max-val','0','--data-stem',DATA_STEM,'--output','ddp_ab_'+str(nproc)]
    if nproc==2:
        cmd=['accelerate','launch','--num_processes','2','--multi_gpu','-m','llm_training.run_train']+args
    else:
        cmd=['accelerate','launch','--num_processes','1','-m','llm_training.run_train']+args
    e={**os.environ,'PYTHONPATH':WORKDIR+'/src/llm','PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}
    r=subprocess.run(cmd,cwd=WORKDIR,capture_output=True,text=True,env=e)
    if r.returncode!=0:
        print('nproc=',nproc,'FAILED'); print(r.stderr[-2000:]); return None
    secs=[float(x) for x in re.findall(r'([0-9.]+)s/step', r.stdout)]
    steady=statistics.median(secs[1:]) if len(secs)>1 else (secs[0] if secs else None)
    print('nproc=',nproc,' steps=',secs,' steady=',round(steady,2),'s/step'); return steady
if DDP:
    s2=_run(2); s1=_run(1)
    if s1 and s2:
        speedup=(2/s2)/(1/s1)
        print('')
        print('2-GPU',round(s2,2),'s/step (2 ex/step)  vs  1-GPU',round(s1,2),'s/step (1 ex/step)')
        print('==> DDP effective speedup =',round(speedup,2),'x')
        print('GO with DDP=True' if speedup>1.3 else 'speedup marginal -> single-GPU is fine too')
else:
    print('DDP=False -> single-GPU; skip A/B.')


In [ ]:
# Cell 6.6 — AUTO-RESUME from the Hub (no manual download / Kaggle Dataset / RESUME flag).
# Each session: if CKPT_REPO holds a checkpoint, pull it and continue automatically;
# otherwise start fresh. Re-running the notebook top-to-bottom just picks up where the
# last session stopped. This cell SETS `RESUME` from what it finds (overrides Cell 1).
# Done ONCE in the kernel before accelerate launch, so the 2 DDP ranks read staged local
# files (no concurrent-download race).
import os, shutil, glob
dst = f"{WORKDIR}/runs/{OUTPUT}"
os.makedirs(dst, exist_ok=True)

def _has_state(d):
    return os.path.exists(f"{d}/checkpoint/trainer_state.pt")

if CKPT_REPO:
    # Hub mode (recommended): pull checkpoint/ (latest, for resume) + best/ (lowest-val,
    # for serving). Works across Kaggle accounts with no Dataset upload — the Hub is the
    # shared store. An empty/first-run repo simply yields no checkpoint -> fresh start.
    from huggingface_hub import snapshot_download
    try:
        snapshot_download(repo_id=CKPT_REPO, repo_type="model",
                          allow_patterns=["checkpoint/*", "best/*"], local_dir=dst)
        print("Hub pull OK:", CKPT_REPO)
    except Exception as e:
        print("Hub pull skipped (", repr(e), ") — will start fresh if no local checkpoint")
elif RESUME:
    # Legacy fallback: stage from a Kaggle Dataset (zip or folder) set in RESUME_DATASET.
    src = RESUME_DATASET
    zips = glob.glob(f"{src}/**/*.zip", recursive=True)
    if zips:
        print("unzip", zips[0], "->", dst); shutil.unpack_archive(zips[0], dst)
    elif os.path.isdir(src):
        for sub in ("checkpoint", "best"):
            s = os.path.join(src, sub)
            if not os.path.isdir(s):
                s = next((q for q in glob.glob(f"{src}/**/{sub}", recursive=True)), None)
            if s and os.path.isdir(s):
                shutil.copytree(s, os.path.join(dst, sub), dirs_exist_ok=True)
                print("copied", s, "->", os.path.join(dst, sub))

# Decide resume vs fresh from what actually landed on disk — not a manual flag.
RESUME = _has_state(dst)
if RESUME:
    import torch
    st = torch.load(f"{dst}/checkpoint/trainer_state.pt", map_location="cpu")
    bv = st.get("best_val")
    bv = f"{bv:.4f}" if isinstance(bv, (int, float)) else bv
    print(f"✅ AUTO-RESUME: continuing from update {st.get('update')}/{MAX_STEPS} "
          f"(epoch {st.get('epoch')}, best_val {bv})")
    print("   checkpoint files:", os.listdir(f"{dst}/checkpoint"))
    if os.path.isdir(f"{dst}/best"):
        print("   best/ present (for serving):", os.listdir(f"{dst}/best"))
else:
    print("No checkpoint found — fresh run (first session). RESUME=False.")


In [ ]:
# Cell 7 - train (E4B QLoRA, Unsloth). DDP=True -> accelerate launch 2 procs (~2x).
# Checkpoints every SAVE_EVERY + on timeout/crash; best/ = lowest val. Rank0 saves.
import subprocess, sys, os
args = ['--model', MODEL, '--engine', ENGINE, '--max-steps', str(MAX_STEPS),
        '--rank', str(RANK), '--targets', TARGETS, '--grad-accum', str(GRAD_ACCUM),
        '--max-seq', str(MAX_SEQ), '--eval-every', str(EVAL_EVERY), '--max-val', str(MAX_VAL),
        '--save-every', str(SAVE_EVERY), '--data-stem', DATA_STEM, '--output', OUTPUT]
if NO_4BIT: args.append('--no-4bit')
if RESUME:  args.append('--resume')
if DDP:
    cmd = ['accelerate','launch','--num_processes','2','--multi_gpu','-m','llm_training.run_train'] + args
else:
    cmd = [sys.executable,'-m','llm_training.run_train'] + args
print('>', ' '.join(cmd))
subprocess.run(cmd, check=True, cwd=WORKDIR,
               env={**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm',
                    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'})


In [ ]:
# Cell 8 — package adapter for download (OPTIONAL; run after training OR a timeout).
# Resume is now AUTOMATIC via the Hub (Cell 6.6), so you do NOT need this zip to continue
# — it's just a convenient local copy of the final/best adapter to pull onto the 4060.
# runs/OUTPUT contains: best/ (lowest-val -> SERVE this) + checkpoint/ (latest, for resume)
# + the final adapter. With CKPT_REPO set, best/ and checkpoint/ are already on the Hub.
import shutil, os
run_dir = f"{WORKDIR}/runs/{OUTPUT}"
print("run dir contents:", os.listdir(run_dir) if os.path.isdir(run_dir) else "MISSING")
out_zip = f"/kaggle/working/{OUTPUT}"
shutil.make_archive(out_zip, "zip", run_dir)
sz = os.path.getsize(out_zip + ".zip") / 1e6
print(f"\n✅ {out_zip}.zip ({sz:.1f} MB) — download from the Output panel (optional).")
print("   SERVE  -> the best/ folder inside (lowest val), or pull best/ from CKPT_REPO.")
print("   RESUME -> AUTOMATIC next session: just re-run the notebook (Cell 6.6 pulls the")
print("             latest checkpoint from CKPT_REPO and continues). No upload needed.")
